# FRED Series Bronze Ingestion

Thin notebook entrypoint for FRED Bronze observations and metadata ingestion.

This notebook delegates HTTP request handling, pagination, normalization, deduplication, and Delta merge logic to `src/lakehouse`.

Execution modes:

- `backfill`: explicit inclusive observation date range
- `incremental`: observation-window re-pull for phase-1 correctness

The target tables must already exist. Run `00_platform_setup_catalog_schema.ipynb` first.

In [ ]:
import sys
import uuid
from pathlib import Path


def add_repo_src_to_path() -> str:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        src_dir = candidate / "src"
        if (src_dir / "lakehouse" / "__init__.py").exists():
            src_dir_str = str(src_dir)
            if src_dir_str not in sys.path:
                sys.path.insert(0, src_dir_str)
            return src_dir_str

    raise RuntimeError("Could not locate the repository src directory from the current notebook path")


REPO_SRC = add_repo_src_to_path()

from lakehouse.pipelines.bronze import run_fred_bronze_ingestion

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("series_ids", "CPIAUCSL,FEDFUNDS,GDP")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "90")
dbutils.widgets.text("run_id", str(uuid.uuid4()))
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("secret_key", "fred_api_key")


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
target_table = f"{catalog}.brz_macro.raw_fred_series"
metadata_table = f"{catalog}.brz_macro.raw_fred_series_metadata"
secret_scope = dbutils.widgets.get("secret_scope").strip()
secret_key = dbutils.widgets.get("secret_key").strip() or "fred_api_key"

if not secret_scope:
    raise ValueError(
        "secret_scope is required so the notebook can read the FRED API key from Databricks secrets"
    )

api_key = dbutils.secrets.get(secret_scope, secret_key)

result = run_fred_bronze_ingestion(
    spark=spark,
    raw_series_ids=dbutils.widgets.get("series_ids").strip(),
    mode=dbutils.widgets.get("mode").strip().lower(),
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
    run_id=dbutils.widgets.get("run_id").strip(),
    api_key=api_key,
    target_table=target_table,
    metadata_table=metadata_table,
    display_fn=display,
)

result.as_dict()
